# M9 · Build BI marts

**Goal:** populate the three BI marts that feed the dashboard:
1. `marts.mart_fleet_daily` — day-grain executive KPI (sql/30)
2. `marts.mart_vehicle_monthly` — month-grain vehicle rollup (sql/31)
3. `marts.mart_tenant_monthly_summary` — month-grain tenant rollup of #1 + #2 (sql/32)

**Order matters.** Tenant summary depends on the other two, so it runs last.

**Recompute scope:** for a backfill we pass *all* dates / months currently represented in the warehouse. For incremental runs the orchestrator passes only `:touched_dates` / `:touched_months`.

In [ ]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, run_sql_file, transaction
from sqlalchemy import text
import pandas as pd

s = settings()
print(f'DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}')

## 2. Inputs — derive the touched dates/months from the warehouse facts

In [ ]:
# A backfill recomputes every date/month that has any fact rows.
with get_engine().connect() as conn:
    dates = pd.read_sql(text('''
        SELECT begin_path_time::DATE AS d FROM warehouse.fact_trip
        UNION SELECT notification_date FROM warehouse.fact_notification
        UNION SELECT maintenance_date FROM warehouse.fact_maintenance
        UNION SELECT fueling_date FROM warehouse.fact_fueling
        UNION SELECT event_time::DATE FROM warehouse.fact_harsh_event
        UNION SELECT stop_start::DATE FROM warehouse.fact_stop
        UNION SELECT begin_path_time::DATE FROM warehouse.fact_overspeed
    '''), conn)
TOUCHED_DATES = sorted(dates['d'].dropna().astype(str).tolist())
TOUCHED_MONTHS = sorted({d[:7] for d in TOUCHED_DATES})
print(f'dates: {len(TOUCHED_DATES):,}   months: {len(TOUCHED_MONTHS)}')
print('first/last month:', TOUCHED_MONTHS[:1], '...', TOUCHED_MONTHS[-1:])

## 3. Execute (in dependency order)

In [ ]:
from accent_fleet.pipeline.run_log import begin_run, end_run
import time

run_id = begin_run(mode='notebook:05_build_bi_marts')
t0 = time.time(); written = {}
try:
    with transaction() as conn:
        r = run_sql_file(conn, '30_mart_fleet_daily.sql',
                         params={'touched_dates': TOUCHED_DATES, 'etl_run_id': run_id})
        written['fleet_daily'] = r.rowcount or 0
        r = run_sql_file(conn, '31_mart_vehicle_monthly.sql',
                         params={'touched_months': TOUCHED_MONTHS, 'etl_run_id': run_id})
        written['vehicle_monthly'] = r.rowcount or 0
        # tenant_summary depends on the two above — same transaction.
        r = run_sql_file(conn, '32_mart_tenant_monthly_summary.sql',
                         params={'touched_months': TOUCHED_MONTHS, 'etl_run_id': run_id})
        written['tenant_summary'] = r.rowcount or 0
    end_run(run_id, status='success', rows_loaded=sum(written.values()))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc)); raise
print(f'done in {time.time()-t0:.1f}s', written)

## 4. Inspect

In [ ]:
with get_engine().connect() as conn:
    head_fd = pd.read_sql(text('SELECT * FROM marts.mart_fleet_daily ORDER BY fleet_date DESC LIMIT 5'), conn)
    head_vm = pd.read_sql(text('SELECT * FROM marts.mart_vehicle_monthly ORDER BY year_month DESC, total_distance_km DESC LIMIT 5'), conn)
    head_ts = pd.read_sql(text('SELECT * FROM marts.mart_tenant_monthly_summary ORDER BY year_month DESC LIMIT 5'), conn)
display(head_fd); display(head_vm); display(head_ts)